In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
import torch.nn.functional as F

device_mps = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


In [2]:
data = pd.read_csv('/Users/libinvarghese/Documents/Jesna/Git_ML/ML_algorithms_Scratch/data/Iris.csv')

In [3]:
data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [4]:
X = data.iloc[:,1:5]
y = data.Species

In [5]:

map ={'Iris-setosa':0,'Iris-versicolor':1,'Iris-virginica':2}
y =y.map(map)

In [6]:
X = torch.tensor(X.values,dtype =torch.float)
y = torch.tensor(y.values,dtype=torch.float)
torch.manual_seed(123)
shuffle_idx = torch.randperm(X.size(0),dtype =torch.int)
X,y = X[shuffle_idx],y[shuffle_idx]

perc80 = int(shuffle_idx.size(0)*0.8)
X_train,X_test = X[shuffle_idx[:perc80]],X[shuffle_idx[perc80:]]
y_train,y_test = y[shuffle_idx[:perc80]],y[shuffle_idx[perc80:]]

mu,sigma = X_train.mean(dim =0),X_train.std(dim=0)

X_train =(X_train-mu)/sigma
X_test =(X_test-mu)/sigma



In [10]:
from softmax_regression import LogisticRegression2
def comp_accuracy(labels,predicted):
    accuracy = torch.sum(labels.float()==predicted.float())/labels.size(0)
    return accuracy

In [8]:
model2 = LogisticRegression2(num_features=4,num_classes=3).to(device_mps)
optimizer = torch.optim.SGD(model2.parameters(),lr =0.1)

In [11]:
X_train = X_train.to(device_mps)
y_train=y_train.to(device_mps)
X_test =X_test.to(device_mps)
y_test=y_test.to(device_mps)

num_epcho=50

for epoch in range(num_epcho):
    logits,probs = model2.forward(X_train)
    loss = F.cross_entropy(logits,y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    acc = comp_accuracy(y_train,torch.argmax(probs,dim =1))
    print('Epoch: %03d' % (epoch + 1), end="")
    print(' | Train ACC: %.3f' % acc, end="")
    print(' | Cost: %.3f' % F.cross_entropy(logits, y_train))


Epoch: 001 | Train ACC: 0.800 | Cost: 1.088
Epoch: 002 | Train ACC: 0.800 | Cost: 1.077
Epoch: 003 | Train ACC: 0.800 | Cost: 1.066
Epoch: 004 | Train ACC: 0.800 | Cost: 1.055
Epoch: 005 | Train ACC: 0.792 | Cost: 1.044
Epoch: 006 | Train ACC: 0.792 | Cost: 1.033
Epoch: 007 | Train ACC: 0.792 | Cost: 1.022
Epoch: 008 | Train ACC: 0.792 | Cost: 1.012
Epoch: 009 | Train ACC: 0.792 | Cost: 1.001
Epoch: 010 | Train ACC: 0.792 | Cost: 0.992
Epoch: 011 | Train ACC: 0.792 | Cost: 0.982
Epoch: 012 | Train ACC: 0.792 | Cost: 0.973
Epoch: 013 | Train ACC: 0.792 | Cost: 0.964
Epoch: 014 | Train ACC: 0.792 | Cost: 0.956
Epoch: 015 | Train ACC: 0.792 | Cost: 0.947
Epoch: 016 | Train ACC: 0.792 | Cost: 0.940
Epoch: 017 | Train ACC: 0.792 | Cost: 0.933
Epoch: 018 | Train ACC: 0.792 | Cost: 0.926
Epoch: 019 | Train ACC: 0.792 | Cost: 0.919
Epoch: 020 | Train ACC: 0.792 | Cost: 0.913
Epoch: 021 | Train ACC: 0.792 | Cost: 0.907
Epoch: 022 | Train ACC: 0.792 | Cost: 0.901
Epoch: 023 | Train ACC: 0.792 | 

/var/folders/7b/pb4y2ycs0s5fhd1y7ddbk2500000gn/T/ipykernel_3591/3663172589.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print(' | Cost: %.3f' % F.cross_entropy(logits, y_train))


In [12]:
predict_logit,pred_prob = model2.forward(X_test)
acc_test = comp_accuracy(y_test,torch.argmax(pred_prob,dim =1))
print(' Test ACC: %.3f' % acc_test, end="")


 Test ACC: 0.733

In [14]:
print(model2.linear.weight)
print(model2.linear.bias)

Parameter containing:
tensor([[-0.3506,  0.3314, -0.4831, -0.4653],
        [-0.0105, -0.3140,  0.0665,  0.0054],
        [ 0.3611, -0.0174,  0.4166,  0.4599]], device='mps:0',
       requires_grad=True)
Parameter containing:
tensor([ 0.0282, -0.0654,  0.0372], device='mps:0', requires_grad=True)
